# 06 — Fidelidade quantitativa do Grad-CAM (IoU)

A Etapa 4 tem duas partes. A parte 1 (`src/extract_gradcam.py`,
notebook `05`) extrai os mapas do Grad-CAM e os compara com as anotações
clínicas **por inspeção visual**. Esta é a parte 2: reduzir essa comparação
a **um número por batimento**.

O número é a **Interseção sobre União (IoU)** entre o mapa do Grad-CAM
binarizado (`cam_interp >= 0.5`, já que o mapa vem normalizado em `[0, 1]`)
e a `mascara_clinica` do LUDB — a união P+QRS+T amostra a amostra. Toda a
lógica está em `src/fidelity_ludb.py`; aqui só carregamos os mapas já
extraídos e as máscaras do dataset e cruzamos os dois.

Execute antes, na raiz do repositório:

```bash
python src/train_cnn.py
python src/extract_gradcam.py
```


## 1. Carregando os dados pré-computados

Nenhum modelo é carregado: `extract_gradcam.py` já fez o forward+backward
uma vez e salvou `cam_interp` `(n, 234)`. As máscaras clínicas vêm do
dataset processado do LUDB, na mesma ordem de índice (o pipeline não
embaralha em nenhum passo).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# O codigo do projeto vive em src/ (nao ha pacote instalavel).
sys.path.insert(0, str(Path.cwd().parent / "src"))
import config
import fidelity_ludb as fidelity
import beat_io

RUN_DIR = config.MODELS_DIR / "cnn1d_mitbih"
GRADCAM_FILE = RUN_DIR / "gradcam_ludb" / "ludb_gradcam.npz"

gcam = np.load(GRADCAM_FILE, allow_pickle=True)
cam_interp = gcam["cam_interp"]   # (n, 234), normalizado em [0, 1]
y_true = gcam["y_true"].astype(int)
y_pred = gcam["y_pred"].astype(int)
ritmos = gcam["ritmos"]

ludb = beat_io.load_beat_dataset(config.LUDB_PROCESSED_FILE)
mascara_clinica = ludb["mascara_clinica"]
mascaras_estrutura = {
    "P": ludb["mascara_p"],
    "QRS": ludb["mascara_qrs"],
    "T": ludb["mascara_t"],
}
cobertura_acaso = ludb["metadata"]["cobertura_media_mascara_clinica"]

assert cam_interp.shape == mascara_clinica.shape
print(f"{cam_interp.shape[0]} batimentos | janela {cam_interp.shape[1]} amostras")
print(f"cobertura media da mascara clinica (acaso): {cobertura_acaso:.4f}")

## 2. IoU por batimento e médias

`fidelity.fidelidade` binariza o mapa, calcula o IoU de cada batimento
contra a máscara clínica e agrega: geral, por classe **prevista** e por
classe **verdadeira**. A quebra por classe prevista importa porque o mapa é
normalizado pelo próprio máximo — num verdadeiro negativo, ativação fraca e
ruidosa ainda vira um pico "quente", então o IoU nesses batimentos mede
outra coisa.

In [ ]:
res = fidelity.fidelidade(
    cam_interp,
    mascara_clinica,
    y_pred=y_pred,
    y_true=y_true,
    mascaras_por_estrutura=mascaras_estrutura,
    cobertura_acaso=cobertura_acaso,
)
iou = res["iou_por_batimento"]
print(fidelity.formata_fidelidade("LUDB", res))

A leitura do número: um IoU médio **acima** da cobertura da máscara
(`esperado ao acaso`) é a evidência mínima de que o Grad-CAM olha para as
estruturas anotadas e não para pontos arbitrários. `acima do acaso ≤ 0`
significa que o mapa não bate a linha de base aleatória.

## 3. Distribuição do IoU

A média esconde a forma. Aqui vemos a distribuição inteira, separada por
classe prevista, com a referência ao acaso marcada.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))

eixos[0].hist(iou[~np.isnan(iou)], bins=40, color="#7f8c8d")
eixos[0].axvline(cobertura_acaso, color="k", ls="--", lw=1, label="esperado ao acaso")
eixos[0].axvline(np.nanmean(iou), color="#c0392b", lw=1.5, label="IoU medio")
eixos[0].set_xlabel("IoU (mapa binarizado x mascara clinica)")
eixos[0].set_ylabel("n batimentos")
eixos[0].set_title("Distribuicao do IoU")
eixos[0].legend(fontsize=8)

for classe, cor, rotulo in [(0, "#2980b9", "previu normal"), (1, "#c0392b", "previu arritmia")]:
    eixos[1].hist(
        iou[(y_pred == classe) & ~np.isnan(iou)], bins=40, alpha=0.6,
        color=cor, label=rotulo, density=True,
    )
eixos[1].axvline(cobertura_acaso, color="k", ls="--", lw=1)
eixos[1].set_xlabel("IoU")
eixos[1].set_ylabel("densidade")
eixos[1].set_title("IoU por classe prevista")
eixos[1].legend(fontsize=8)

for eixo in eixos:
    eixo.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 4. IoU por ritmo

Reaproveitando `metrics.por_grupo` — a mesma quebra por ritmo do teste
(`04_teste_ludb.ipynb`), agora sobre o IoU em vez do acerto. Ritmos onde o
modelo erra mas o Grad-CAM ainda foca P/QRS/T contam uma história diferente
de ritmos onde o mapa se dispersa.

In [ ]:
import pandas as pd

linhas = []
for ritmo in np.unique(ritmos):
    m = ritmos == ritmo
    linhas.append({
        "ritmo": ritmo,
        "n": int(m.sum()),
        "classe": int(y_true[m][0]),
        "iou_medio": float(np.nanmean(iou[m])),
        "acerto": float((y_pred[m] == y_true[m]).mean()),
    })
tabela = pd.DataFrame(linhas).sort_values("n", ascending=False).reset_index(drop=True)
tabela

## 5. Overlay: onde o mapa e a anotação concordam

Para os batimentos de maior e menor IoU: o ECG ao fundo, a máscara clínica
(verde) e a região binarizada do Grad-CAM (vermelho hachurado) sobrepostas.
A interseção das duas faixas é o numerador do IoU; a união, o denominador.

In [ ]:
tempo = (np.arange(cam_interp.shape[1]) - config.WINDOW_LEFT) / config.TARGET_FS * 1000
cam_bin = fidelity.binariza_cam(cam_interp)


def desenha(idx, eixo):
    eixo.plot(tempo, ludb["series"][idx], color="#2c3e50", lw=1)
    eixo.fill_between(tempo, 0, 1, where=mascara_clinica[idx] > 0,
                      transform=eixo.get_xaxis_transform(),
                      color="#27ae60", alpha=0.20, label="mascara clinica")
    eixo.fill_between(tempo, 0, 1, where=cam_bin[idx],
                      transform=eixo.get_xaxis_transform(),
                      facecolor="none", edgecolor="#c0392b", hatch="///",
                      linewidth=0.0, label="Grad-CAM >= 0.5")
    eixo.set_title(
        f"batimento {idx} | real={y_true[idx]} pred={y_pred[idx]} | IoU={iou[idx]:.3f}",
        fontsize=9,
    )
    eixo.set_xlabel("ms em relacao ao pico R")


ordem = np.argsort(np.where(np.isnan(iou), -1, iou))
selecao = {"maior IoU": ordem[-1], "mediano": ordem[len(ordem) // 2], "menor IoU": ordem[0]}

fig, eixos = plt.subplots(1, 3, figsize=(16, 3.6), sharey=True)
for eixo, (rotulo, idx) in zip(eixos, selecao.items()):
    desenha(int(idx), eixo)
eixos[0].legend(fontsize=7, loc="upper right")
fig.tight_layout()
plt.show()

## 6. Próximo passo

O mesmo cálculo em lote, com saída persistida (`ludb_fidelity.npz` +
`ludb_fidelity.json`), está em `src/fidelity_ludb.py`. Falta a análise
de falhas da métrica por tipo de arritmia (Etapa 4) e a redação (Etapa 5).

Uma limitação registrada desde o notebook `05`: o IoU usa o mapa
normalizado pelo máximo de cada batimento, então não distingue "ativação
forte e bem localizada" de "ativação fraca e ruidosa". Acompanhar o IoU da
magnitude bruta do pico de ativação exige salvar o mapa antes da
normalização em `extract_gradcam.py`.